In [4]:
import pandas as pd
import sqlite3

In [3]:
def load_and_validate(file_path, required_column):
    # Load csv and validate structure.
    df = pd.read_csv(file_path)
    
    # Standardize columns
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    
    # Check if required columns exit
    missing = set(required_column) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
    # Check for nulls in key fields
    null_counts = df.isnull().sum(0)
    print(f"Number of nulls: {null_counts[null_counts > 0]}")
    
    return df

# Load all sources
sales = load_and_validate('sales_transactions.csv', ['date', 'store_id', 'product_id', 'revenue'])
stores = load_and_validate('store_locations.csv', ['store_id', 'region'])
products = load_and_validate('product_master.csv', ['product_id', 'category'])
customers = load_and_validate('customers.csv', ['customer_id', 'signup_date'])

# Merge into single analytical dataset
df = sales.merge(stores, on='store_id') \
        .merge(products, on='product_id') \
        .merge(customers, on='customer_id')
        
print(f"Final Dataset: {df.shape}")
df.to_csv('clean_retail_data.csv', index=False)
df.head()

Number of nulls: Series([], dtype: int64)
Number of nulls: Series([], dtype: int64)
Number of nulls: Series([], dtype: int64)
Number of nulls: Series([], dtype: int64)
Final Dataset: (100000, 22)


,transaction_id,date,store_id,product_id,customer_id,quantity,unit_price,discount_pct,revenue,store_name,...,sq_footage,opening_date,product_name,category,unit_cost,launch_date,age,signup_date,segment,region_y
0,1,2023-09-19,104,1110,5925,1,484.66,0.0,484.66,Northpoint Center,...,55000,2017-11-05,Keyboard Model-110,Electronics,166.59,2023-05-25,59,2023-05-17,Standard,South
1,2,2024-01-28,102,1134,7385,1,726.39,15.0,617.43,Westfield Mall,...,62000,2019-07-22,Speaker Model-134,Electronics,296.20,2023-02-05,63,2024-10-24,Standard,East
2,3,2023-09-05,104,1108,9052,1,1220.33,0.0,1220.33,Northpoint Center,...,55000,2017-11-05,Rug Design-108,Home & Garden,560.08,2023-02-23,51,2023-08-22,Premium,South
3,4,2023-10-14,103,1164,7260,3,88.09,0.0,264.27,Eastside Market,...,38000,2020-01-10,Socks Style-164,Clothing,29.78,2023-12-23,23,2023-07-03,Standard,East
4,5,2023-12-12,101,1026,5681,2,317.69,0.0,635.38,Downtown Plaza,...,45000,2018-03-15,Shelf Design-26,Home & Garden,143.47,2023-06-16,30,2023-03-16,Standard,South


In [19]:
# Store & Category Performance
conn = sqlite3.connect('retail_dw.db') # Connecting to the database

query = """
    WITH quarterly AS(
        SELECT
            strftime('%Y', date) AS yr,
            CASE
                WHEN CAST(strftime('%m', date) AS INT) BETWEEN 1 AND 3 THEN 1
                WHEN CAST(strftime('%m', date) AS INT) BETWEEN 4 AND 6 THEN 2
                WHEN CAST(strftime('%m', date) AS INT) BETWEEN 7 AND 9 THEN 3
                ELSE 4
            END AS qtr,
            region,
            category,
            SUM(revenue) AS revenue
        FROM sales s
        JOIN stores st ON s.store_id = st.store_id
        JOIN products p ON s.product_id = p.product_id
        GROUP BY yr, qtr, region, category
    ),
    with_lag AS(
        SELECT
            yr,
            qtr,
            region,
            category,
            revenue,
            LAG(revenue) OVER (PARTITION BY region, category ORDER BY yr, qtr) AS prev_quarter
        FROM quarterly
    )
    SELECT
        region,
        category,
        revenue,
        prev_quarter,
        ROUND(
            (revenue - prev_quarter) * 100 / prev_quarter, 2
        ) AS growth_pct
    FROM with_lag
    WHERE yr = '2024' AND qtr = 2;
"""

df_query = pd.read_sql_query(query, conn)
print(df_query)



               region       category  revenue  prev_quarter  growth_pct
0      Downtown Plaza       Clothing     1305          1245         4.0
1      Downtown Plaza    Electronics     1320          1830       -27.0
2      Downtown Plaza  Home & Garden     1005          1185       -15.0
3     Eastside Market       Clothing     1345          1715       -21.0
4     Eastside Market    Electronics     1770          1840        -3.0
5     Eastside Market  Home & Garden      885          1415       -37.0
6   Northpoint Center       Clothing     1130          1435       -21.0
7   Northpoint Center    Electronics     1380          1645       -16.0
8   Northpoint Center  Home & Garden      980          1070        -8.0
9      Westfield Mall       Clothing     1280          1330        -3.0
10     Westfield Mall    Electronics     1420          1975       -28.0
11     Westfield Mall  Home & Garden      950          1245       -23.0
